# Declared Workflows (Offline)

Plan and run composable workflows with local models and tensors. This notebook runs in CI.

In [ ]:
import torch
from torch import nn
from tensordict import TensorDict

torch.manual_seed(0)


class TinyConceptModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(10, 10)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(10, 2)

    def forward(self, input):
        return self.linear2(self.relu(self.linear1(input)))


class FeatureMap(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 12)

    def forward(self, input):
        return self.linear(input).relu().reshape(-1, 3, 2, 2)


class TinyFeatureModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = FeatureMap()
        self.head = nn.Linear(12, 2)

    def forward(self, input):
        features = self.features(input)
        return self.head(features.flatten(1))


concept_model = TinyConceptModel().eval()
dimension_model = TinyFeatureModel().eval()
examples = torch.randn(6, 10)
labels = torch.tensor([1, 1, 1, 0, 0, 0])

## Concept-conditioned attribution: two model passes

The first LRP method publishes concept-example relevance. `ConceptSelection` turns it and the labels into a named TensorDict value, and `ChannelConditionedLRP` reads that value during the second pass. No callback or bound-hook state crosses the boundary.

In [ ]:
from tdhook.attribution import LRP
from tdhook.concepts import ChannelConditionedLRP, ConceptSelection
from tdhook.workflow import Workflow

concept_workflow = Workflow(
    LRP(
        input_modules=["linear1"],
        attribution_key=("attributions", "concept_examples"),
        warn_on_missing_rule=False,
    ),
    ConceptSelection(("attributions", "concept_examples", "linear1")),
    ChannelConditionedLRP(LRP(warn_on_missing_rule=False), condition_module="linear1"),
)
concept_artifacts = TensorDict({"input": examples, "concept_labels": labels}, batch_size=[len(examples)])
assert len(concept_workflow.steps) == 3
tuple(type(step).__name__ for step in concept_workflow.steps)

In [ ]:
concept_result = concept_workflow(concept_model, concept_artifacts)
selection = concept_result[("metrics", "concept_selection")]
conditioned = concept_result[("attributions", "conditioned", "input")]
print("selected channel:", selection["channel"][0].item())
print("conditioned relevance shape:", tuple(conditioned.shape))

## Conditioned intrinsic dimension: one model pass

Only activation capture runs the model. Channel selection, TwoNN estimation, and summary are ordinary TensorDict operators. Plotting or domain rendering belongs downstream of these stable artifacts.

In [ ]:
from tdhook.dimension import channel_conditioned_samples, conditioned_dimension_workflow
from tdhook.latent import ActivationCaching
from tdhook.latent.dimension_estimation import TwoNnDimensionEstimator

dimension_workflow = conditioned_dimension_workflow(
    ActivationCaching("features", cache_key=("activations", "cache")),
    "features",
    channel_conditioned_samples,
    TwoNnDimensionEstimator(),
)
dimension_artifacts = TensorDict({"input": examples}, batch_size=[len(examples)])
assert len(dimension_workflow.steps) == 4
tuple(type(step).__name__ for step in dimension_workflow.steps)

In [ ]:
dimension_result = dimension_workflow(dimension_model, dimension_artifacts)
samples = dimension_result[("activations", "samples")].data
dimensions = dimension_result[("metrics", "dimension")].data
summary = dimension_result[("metrics", "dimension_summary")].data
print("samples:", tuple(samples.shape), "dimensions:", tuple(dimensions.shape))
summary

## Disk-backed activation caches

A workflow can publish TensorDict-native memory-mapped storage without materializing an in-memory copy. Preallocate every captured key with its final shape, dtype, and device before creating the memory map. Because memory-mapped TensorDicts have a fixed, locked structure, configure `clear_cache=False`; captures then update the existing leaves in place.

In [ ]:
import tempfile
from pathlib import Path

from tensordict import MemoryMappedTensor

cache_root = Path(tempfile.mkdtemp(prefix="tdhook-activation-cache-"))
cache_path = cache_root / "activations"
disk_cache = TensorDict(
    {("fwd", "features"): torch.empty(len(examples), 3, 2, 2)},
    batch_size=[],
).memmap(cache_path)
disk_caching = ActivationCaching(
    r"features$",
    cache=disk_cache,
    clear_cache=False,
    use_nested_keys=True,
    cache_key=("activations", "cache"),
)
disk_artifacts = TensorDict({"input": examples}, batch_size=[len(examples)])
disk_result = Workflow(disk_caching)(dimension_model, disk_artifacts)
published = disk_result[("activations", "cache", "fwd", "features")]
reloaded = TensorDict.load_memmap(cache_path)

assert isinstance(published, MemoryMappedTensor)
assert published.data_ptr() == disk_cache[("fwd", "features")].data_ptr()
torch.testing.assert_close(reloaded[("fwd", "features")], published)
print("memory-mapped cache:", cache_path)

The caller owns `cache_path` and its lifetime; TDHook neither creates the path nor deletes it. Published leaves are mutable views of the same storage, so a later execution using `disk_cache` changes earlier published views. Use a separate path for an immutable snapshot. Reload with `TensorDict.load_memmap(cache_path)`, and remove the directory only after all published and reloaded views are no longer needed.

In [ ]:
import shutil

del reloaded, published, disk_result, disk_artifacts, disk_caching, disk_cache
shutil.rmtree(cache_root)
assert not cache_root.exists()